# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Haryomhidhe/Machine-learning-flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and  load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My rule: A page is scored as an opportunity only if it has real search demand (impressions > 100 for the month). Among those pages, if CTR is below 0.15%, that means people see it but don't click, so it gets flagged as low_ctr_visible_page. If avg_position is worse than 10, the page isn't ranking well despite demand, so it gets flagged as poor_position. If both apply, it's the highest priority opportunity, flagged as low_ctr_and_poor_position. Pages with real demand that don't hit either problem are performing_well and left alone. Pages with low demand (under 100 impressions) are low_demand, meaning there isn't enough traffic yet to justify reviewing them

In [35]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import pandas as pd
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

In [36]:
monthly = con.sql(f"""
SELECT content_hash_id,
  SUM(gsc_impressions) AS total_impressions,
  SUM(gsc_clicks) AS total_clicks,
  AVG(gsc_avg_position) AS avg_position
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-02/*.parquet')
GROUP BY content_hash_id
""").df()

monthly.shape

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(321546, 4)

Signal 1: Position vs CTR (ties to the CTR-vs-position flag). Verdict: CONFIRMED.
I grouped pages with more than 100 impressions in February 2026 by average position. CTR fell at every step: 0.41% at positions 1-3 (n = 11,074), 0.34% at 4-10 (n = 40,985), 0.31% at 11-20 (n = 16,706), and 0.15% at 20+ (n = 11,358). This is an observed, directional pattern: pages ranked lower get a smaller share of clicks. This means low CTR is partly explained by position, so my rule’s “low CTR and poor position” group may count one problem twice. The drop from 1-3 to 11-20 is small, though, so position alone does not explain all of the low CTR.

In [37]:
t = monthly[monthly['total_impressions'] > 100].copy()
t['position_bucket'] = pd.cut(t['avg_position'], [0, 3, 10, 20, 10000],
                              labels=['1-3', '4-10', '11-20', '20+'])
sig1 = t.groupby('position_bucket', observed=True).agg(
    n=('content_hash_id', 'count'),
    clicks=('total_clicks', 'sum'),
    impressions=('total_impressions', 'sum'))
sig1['ctr_pct'] = sig1['clicks'] / sig1['impressions'] * 100
sig1

,n,clicks,impressions,ctr_pct
position_bucket,,,,
1-3,11074,136800.0,33408697.0,0.409474
4-10,40985,343576.0,102477704.0,0.335269
11-20,16706,69201.0,22252028.0,0.310987
20+,11358,30491.0,20250234.0,0.150571


In [38]:
d = monthly[monthly['total_impressions'] > 100].copy()
d['ctr'] = d['total_clicks'] / d['total_impressions'] * 100
print("share of pages with CTR below 2%:", round((d['ctr'] < 2).mean() * 100, 1), "%")
print(d['ctr'].quantile([0.1, 0.25, 0.5, 0.75, 0.9]).round(3))

share of pages with CTR below 2%: 99.2 %
0.10    0.000
0.25    0.000
0.50    0.139
0.75    0.385
0.90    0.709
Name: ctr, dtype: float64


In [39]:
for cut in [0.1, 0.15, 0.5]:
    print(cut, round((d['ctr'] < cut).mean() * 100, 1), "% of pages flagged")

0.1 44.9 % of pages flagged
0.15 51.3 % of pages flagged
0.5 82.6 % of pages flagged


In [40]:
fair = con.sql(f"""
SELECT content_hash_id,
  SUM(gsc_sum_position)::FLOAT / NULLIF(SUM(gsc_impressions), 0) AS fair_pos
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-02/*.parquet')
GROUP BY content_hash_id
""").df()
chk = monthly.merge(fair, on='content_hash_id')
chk = chk[chk['total_impressions'] > 100]
print("pages whose poor-position flag changes:",
      ((chk['avg_position'] > 10) != (chk['fair_pos'] > 10)).sum(), "of", len(chk))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

pages whose poor-position flag changes: 2585 of 80124


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [41]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df_score = con.sql(f"""
SELECT
  content_hash_id,
  SUM(gsc_impressions) as total_impressions,
  SUM(gsc_clicks) as total_clicks,
  AVG(gsc_avg_position) as avg_position,
  CASE WHEN SUM(gsc_impressions) > 0
    THEN (SUM(gsc_clicks)::FLOAT / SUM(gsc_impressions)) * 100
    ELSE 0 END as ctr
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-02/*.parquet')
GROUP BY content_hash_id
""").df()

def score_page(row):
    if row['total_impressions'] <= 100:
        return 0, 'low_demand', 'leave_as_is'

    low_ctr = row['ctr'] < 0.15
    poor_position = row['avg_position'] > 10

    if not low_ctr and not poor_position:
        return 0, 'performing_well', 'leave_as_is'

    severity = 0
    if low_ctr:
        severity += 2
    if poor_position:
        severity += 1

    score = row['total_impressions'] * severity

    if low_ctr and poor_position:
        reason = 'low_ctr_and_poor_position'
        action = 'review_and_refresh'
    elif low_ctr:
        reason = 'low_ctr_visible_page'
        action = 'review_ctr'
    else:
        reason = 'poor_position'
        action = 'review_position'

    return score, reason, action

df_score[['action_score', 'reason_code', 'suggested_action']] = df_score.apply(
    lambda row: pd.Series(score_page(row)), axis=1
)

df_score = df_score.sort_values('action_score', ascending=False)

import os
os.makedirs('work/outputs', exist_ok=True)
df_score.to_csv('work/outputs/baseline_action_score.csv', index=False)

df_score.head(20)

,content_hash_id,total_impressions,total_clicks,avg_position,ctr,action_score,reason_code,suggested_action
70618,content_8e1334d6356668e3,203401.0,2.0,4.967059,0.000983,406802.0,low_ctr_visible_page,review_ctr
102585,content_9c057b66c30a3abb,195648.0,1.0,2.488437,0.000511,391296.0,low_ctr_visible_page,review_ctr
231740,content_fec55986a1868d62,193954.0,0.0,3.844678,0.000000,387908.0,low_ctr_visible_page,review_ctr
182739,content_df47d1b976106de4,85935.0,103.0,18.008694,0.119858,257805.0,low_ctr_and_poor_position,review_and_refresh
231739,content_c9f840183215651b,125035.0,0.0,9.366950,0.000000,250070.0,low_ctr_visible_page,review_ctr
286869,content_84a6bf3578312e90,79986.0,74.0,20.050958,0.092516,239958.0,low_ctr_and_poor_position,review_and_refresh
85226,content_7c6373141eae744a,116407.0,101.0,5.129799,0.086765,232814.0,low_ctr_visible_page,review_ctr
288197,content_ba462518dad435fc,69134.0,39.0,27.753576,0.056412,207402.0,low_ctr_and_poor_position,review_and_refresh
26185,content_2ac8c7995de53cd1,92128.0,4.0,6.192190,0.004342,184256.0,low_ctr_visible_page,review_ctr
187438,content_44f34c0a90047651,90223.0,13.0,5.133139,0.014409,180446.0,low_ctr_visible_page,review_ctr


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [42]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [43]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.